# `ai_parse_document` — Part 1: Multi-Format Extraction & Scoring

## Scope
This notebook benchmarks Databricks `ai_parse_document` (v2.0) on **3 document types**:

| File | Format | Key challenge |
|---|---|---|
| `Invoice.jpg` | Scanned image (JPG) | Typed text + financial table from low-res scan |
| `sample_dashboard.png` | Image / PNG | Figures, KPI numbers, chart data from a raster image |
| `AccidentStatement.pdf` | Native PDF | Multi-column form, handwriting, checkboxes, diagrams |

> **Part 2** will add the LandingAI ADE head-to-head comparison.  
> **Part 3** will cover scanned document preprocessing (`old_articles.pdf`).


## 1. Configuration

In [0]:
%run ../_config/config_unity_catalog

In [0]:
import json, re
import time
from datetime import datetime
from html.parser import HTMLParser
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, FloatType
)

# ── Paths ───────────────────────────────────────────────────────────────────
PATH_VOLUME  = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_IMAGES  = f"/Volumes/{catalog}/{schema}/raw_data/parsed_images"
PATH_RESULTS = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"
dbutils.fs.mkdirs(PATH_IMAGES)
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Part 1: 3 source documents ──────────────────────────────────────────────
# old_articles.pdf (scanned, 0 elements) is handled in Part 3 (preprocessing pipeline)
FILES = {
    "Invoice_jpg"           : "Invoice.jpg",
    "dashboard_png"         : "sample_dashboard.png",
    "AccidentStatement_pdf" : "AccidentStatement.pdf",
}

print(f"Volume : {PATH_VOLUME}")
print(f"Files  : {list(FILES.values())}")


## 2. Parse All Documents with `ai_parse_document` v2.0

We use:
- `version = '2.0'` — latest schema (Jan 2026), based on `elements` array (replaces the legacy `pages` array)
- `descriptionElementTypes = '*'` — in v2.0, `'*'` and `'figure'` produce **the same behaviour**: AI descriptions are generated for **figures only** (descriptions for tables and text blocks are not yet supported)
- `imageOutputPath` — persist rendered page images to the volume for visual inspection or multi-modal RAG

In [0]:
parse_results = {}
parse_timings = {}

for label, filename in FILES.items():
    file_path = f"{PATH_VOLUME}/{filename}"
    print(f"\nParsing [{label}] → {filename}")
    
    t0 = time.time()
    sql = f"""
        WITH parsed AS (
            SELECT
                path,
                ai_parse_document(
                    content,
                    map(
                        'version',                '2.0',
                        'imageOutputPath',        '{PATH_IMAGES}/{label}/',
                        'descriptionElementTypes','*'
                    )
                ) AS parsed
            FROM READ_FILES('{file_path}', format => 'binaryFile')
        )
        SELECT
            path,
            to_json(parsed)                         AS parsed_json,
            parsed:error_status                     AS error_status,
            parsed:metadata:version                   AS version,
            size(cast(parsed:document:elements as array<variant>)) AS element_count
        FROM parsed
    """
    
    df = spark.sql(sql)
    row = df.collect()[0]
    elapsed = round(time.time() - t0, 2)
    parse_timings[label] = elapsed
    
    parsed_data = json.loads(row["parsed_json"])
    parse_results[label] = {
        "filename"      : filename,
        "path"          : row["path"],
        "error_status"  : None if (row["error_status"] is None or str(row["error_status"]).strip('"') in ("null","")) else str(row["error_status"]),
        "version"       : row["version"],
        "element_count" : row["element_count"],
        "parsed"        : parsed_data,
        "elapsed_s"     : elapsed,
    }
    
    status = "" if row["error_status"] is None else f"  {row['error_status']}"
    print(f"   {status}  |  version : {row['version']}  |  elements: {row['element_count']}  |  {elapsed}s")

print("\nAll documents parsed.")

## 2b. HTML Reconstruction

Render each parsed document directly as **HTML** from its `elements` array.

Why HTML instead of Markdown:
- Table elements already contain **valid HTML** (`<table>`, `colspan`, `rowspan`, `<input>`)
- Converting to Markdown is inherently lossy: `colspan` collapses to flat columns,
  `<input type="checkbox" checked>` is silently dropped, nested structure is flattened
- HTML pass-through preserves everything the model extracted — zero conversion loss
- `displayHTML()` renders it natively in the Databricks notebook output

The `elements_to_html()` function:
- Sorts elements by `page_id` → `bbox.y_top` (reading order)
- **Tables**: `_enrich_table_html()` replaces `<input type="checkbox" checked>` → `☑`,
  `<input type="checkbox">` → `☐`, then passes the HTML through unchanged
- **Text / headers**: wrapped in semantic tags (`<h1>`, `<h2>`, `<p>`, etc.)
- **Figures**: AI description in `<figcaption>`, OCR content in `<pre>`
- Page breaks rendered as `<hr>` with page number


In [0]:
import re

# ── Checkbox / input enrichment ────────────────────────────────────────────
def _enrich_table_html(raw: str) -> str:
    """
    Enrich raw table HTML from ai_parse_document:
    - <input type="checkbox" checked> → ☑  (ticked)
    - <input type="checkbox">         → ☐  (unticked)
    - <input type="text">             → removed (blank field placeholder)
    - Adds CSS class to <table> for styling.

    All other HTML (colspan, rowspan, th, td, nested structure) is passed
    through untouched — no conversion loss.
    """
    html = re.sub(r'<input\s+type="checkbox"\s+checked\s*/?>',  '☑', raw,  flags=re.I)
    html = re.sub(r'<input\s+type="checkbox"\s*/?>',             '☐', html, flags=re.I)
    html = re.sub(r'<input\s[^>]*type="text"[^>]*/?>',           '',  html, flags=re.I)
    html = re.sub(r'<input\s+type="text"\s*/?>',                 '',  html, flags=re.I)
    html = html.replace('<table>', '<table class="elem-table">', 1)
    return html

# ── Safe HTML escape for non-table content ─────────────────────────────────
def _esc(s: str) -> str:
    return (s.replace('&', '&amp;')
             .replace('<', '&lt;')
             .replace('>', '&gt;')
             .replace('"', '&quot;'))

# ── Element type → HTML tag ────────────────────────────────────────────────
_TYPE_TAG = {
    'title':          'h1',
    'section_header': 'h2',
    'page_header':    'h3',
    'page_footer':    'footer',
    'caption':        'figcaption',
    'text':           'p',
}

# ── Shared CSS injected once per document ──────────────────────────────────
ELEM_CSS = """
<style>
  .doc-wrap { font-family: Arial, sans-serif; font-size:13px;
              color:#1a1a1a; max-width:1100px; margin:auto; padding:24px 32px; }
  h1.elem-title { font-size:20px; border-bottom:2px solid #333; padding-bottom:6px; margin-bottom:16px; }
  h2.elem-section_header { font-size:14px; color:#1d4ed8; margin:18px 0 4px; text-transform:uppercase; letter-spacing:.04em; }
  h3.elem-page_header  { font-size:12px; color:#555; margin:6px 0 2px; }
  footer.elem-page_footer { font-size:11px; color:#888; display:block; margin:2px 0 6px; }
  p.elem-text { margin:4px 0 8px; line-height:1.5; }
  figcaption.elem-caption { font-style:italic; color:#555; font-size:12px; }
  .elem-table-wrap { margin:8px 0 14px; overflow-x:auto; }
  table.elem-table { border-collapse:collapse; font-size:12px; width:auto; }
  table.elem-table td, table.elem-table th {
    border:1px solid #ccc; padding:4px 10px; vertical-align:top; min-width:60px; }
  table.elem-table th { background:#e8edf4; font-weight:600; }
  figure.elem-figure { background:#f9f9f9; border-left:3px solid #888;
    padding:8px 14px; margin:8px 0 12px; border-radius:0 4px 4px 0; }
  figure.elem-figure figcaption { font-size:12px; color:#444; margin-bottom:4px; }
  pre.ocr-content { font-size:11px; background:#f0f0f0; padding:6px 8px;
    border-radius:3px; white-space:pre-wrap; margin:4px 0 0; }
  hr.page-break { border:none; border-top:1px dashed #bbb; margin:20px 0 4px; }
  p.page-num { font-size:10px; color:#bbb; text-align:center; margin:0 0 14px; }
  .warn { color:#b91c1c; font-style:italic; }
</style>
"""

# ── Core function ───────────────────────────────────────────────────────────
def elements_to_html(elements: list, doc_title: str = '') -> str:
    """
    Render ai_parse_document v2.0 elements directly as HTML.

    Design principle: table content is already valid HTML — pass it through.
    Only non-table elements (text, headers, figures) need wrapping.
    No intermediate Markdown conversion: zero structural loss.
    """
    if not elements:
        return f'<p class="warn">⚠ No elements extracted by ai_parse_document.</p>'

    def sort_key(e):
        bbox = e.get('bbox', [])
        page = bbox[0]['page_id'] if bbox and 'page_id' in bbox[0] else 0
        y    = bbox[0]['coord'][1] if bbox and len(bbox[0].get('coord', [])) >= 2 else 0
        return (page, y)

    parts = []
    if doc_title:
        parts.append(f'<h1 class="elem-title">{_esc(doc_title)}</h1>')

    current_page = -1
    for elem in sorted(elements, key=sort_key):
        bbox        = elem.get('bbox', [])
        page        = bbox[0]['page_id'] if bbox and 'page_id' in bbox[0] else 0
        etype       = elem.get('type', 'text')
        content     = (elem.get('content') or '').strip()
        description = (elem.get('description') or '').strip()

        if page != current_page:
            if current_page >= 0:
                parts.append(f'<hr class="page-break">'
                             f'<p class="page-num">— page {page + 1} —</p>')
            current_page = page

        if etype == 'table':
            # Tables: enrich checkbox/input tags, pass HTML through unchanged
            parts.append(f'<div class="elem-table-wrap">{_enrich_table_html(content)}</div>')

        elif etype == 'figure':
            fig = ['<figure class="elem-figure">']
            if description:
                fig.append(f'<figcaption>📊 {_esc(description)}</figcaption>')
            if content:
                fig.append(f'<pre class="ocr-content">{_esc(content)}</pre>')
            fig.append('</figure>')
            parts.append('\n'.join(fig))

        else:
            tag = _TYPE_TAG.get(etype, 'p')
            cls = f'elem-{etype}'
            if content:
                parts.append(f'<{tag} class="{cls}">{_esc(content)}</{tag}>')

    return ELEM_CSS + '<div class="doc-wrap">\n' + '\n'.join(parts) + '\n</div>'


# ── Generate HTML for all documents ────────────────────────────────────────
doc_htmls = {}

for label, result in parse_results.items():
    elements = result['parsed'].get('document', {}).get('elements', [])
    html = elements_to_html(elements, doc_title=result['filename'])
    doc_htmls[label] = html

    # Save to volume
    out_path = f"{PATH_RESULTS}/{label}.html"
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"[{label}] → {out_path}  ({len(html):,} chars,  {len(elements)} elements)")

print("\nAll HTML outputs saved.")


In [0]:
# ── Preview in notebook ────────────────────────────────────────────────────
# Change PREVIEW_LABEL to: dashboard_png | Invoice_jpg | AccidentStatement_pdf
PREVIEW_LABEL = "AccidentStatement_pdf"

displayHTML(doc_htmls[PREVIEW_LABEL])


In [0]:
PREVIEW_LABEL = "dashboard_png"

displayHTML(doc_htmls[PREVIEW_LABEL])

In [0]:
PREVIEW_LABEL = "Invoice_jpg"

displayHTML(doc_htmls[PREVIEW_LABEL])

## 4. Element-Level Analysis

Extract the `elements` array from each document and build a flat DataFrame to analyse element types, confidence, and content.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

# ── Explicit schema: avoids CANNOT_DETERMINE_TYPE when all values are None ─
# Note: ai_parse_document v2.0 uses 'content' (not 'text') and 'page_id' is
# nested inside bbox[0]['page_id'], not a top-level field.
SCHEMA = StructType([
    StructField("file_label",      StringType(),  False),
    StructField("filename",        StringType(),  False),
    StructField("element_type",    StringType(),  False),
    StructField("page_number",     IntegerType(), True),
    StructField("has_content",     BooleanType(), False),
    StructField("has_description", BooleanType(), False),
    StructField("has_confidence",  BooleanType(), False),
    StructField("confidence",      DoubleType(),  True),   # always None in v2.0 — typed explicitly
    StructField("content_length",  IntegerType(), False),
])

# ── Flatten elements from all files ────────────────────────────────────────
all_elements = []

for label, result in parse_results.items():
    doc      = result["parsed"].get("document", {})
    elements = doc.get("elements", [])

    for elem in elements:
        # page_id is nested inside bbox list in v2.0
        bbox        = elem.get("bbox", [])
        page_number = int(bbox[0]["page_id"]) if bbox and "page_id" in bbox[0] else None

        # v2.0 uses 'content', not 'text'
        content     = elem.get("content") or ""
        description = elem.get("description") or ""
        conf_raw    = elem.get("confidence")          # None in current v2.0 model

        all_elements.append((
            label,
            result["filename"],
            str(elem.get("type", "unknown")),
            page_number,
            bool(content.strip()),
            bool(description.strip()),
            conf_raw is not None,
            float(conf_raw) if conf_raw is not None else None,
            len(content),
        ))

df_elements = spark.createDataFrame(all_elements, schema=SCHEMA)
df_elements.createOrReplaceTempView("elements")

print(f"Total elements across all files: {df_elements.count()}")
display(df_elements.groupBy("file_label", "element_type").count().orderBy("file_label", "count", ascending=[True, False]))

## 5. Per-File Accuracy Scorecard — Redesigned Metrics

### 5 quality-focused metrics

| Metric | Weight | Formula | Measures |
|---|---|---|---|
| `content_density` | **25%** | `min(avg_chars_per_elem / 80, 1.0)` | Are elements rich with content or empty shells? |
| `table_cell_fill` | **25%** | `filled_cells / total_cells` across all tables | Were handwritten/typed values actually captuD inside tables? |
| `kv_extraction_rate` | **20%** | `% of text elements with label:value pairs where value has >2 chars` | Did the model capture values, not just labels? |
| `type_diversity` | **15%** | `min(unique_element_types / 4, 1.0)` | Did it detect multiple structures (text + table + figure + title)? |
| `figure_description_depth` | **15%** | `min(avg_description_len / 150, 1.0)` · `0.5` if no figures | Are visual elements meaningfully described by the AI? |

> **Scoring calibration note**: These thresholds (80 chars, 4 types, 150 chars) are deliberate choices, not universal constants.
> An invoice with short numeric cells will have lower `content_density` than a paragraph-heavy report by design — 
> the score reflects fit for a specific task (form extraction / information retrieval), not raw OCR accuracy.


In [0]:
# ── Helper: count filled cells using the colspan-aware _TableParser ────────
# _TableCellCounter (old version) ignored colspan/rowspan, so a table whose
# header row had colspan="2" was counted as 1-column — same truncation bug
# as the old html_table_to_md.  We now reuse _TableParser (defined in cell 7)
# which already expands colspan/rowspan correctly.
def count_table_cells(html_content: str):
    """Return (total_cells, filled_cells) for an HTML table string.
    Uses the colspan/rowspan-aware _TableParser so merged header cells
    no longer cause data rows to be under-counted."""
    if not html_content or not html_content.strip().startswith('<'):
        return 0, 0
    p = _TableParser()
    p.feed(html_content)
    total = filled = 0
    for row in p.rows:
        for cell in row:
            total += 1
            if len(cell.strip()) > 1:   # >1 char: excludes lone dashes, spaces
                filled += 1
    return total, filled


# ── Main scoring loop ───────────────────────────────────────────────────────
scorecards = []

for label, result in parse_results.items():
    elements   = result['parsed'].get('document', {}).get('elements', [])
    elem_count = len(elements)

    # ── 1. content_density ─────────────────────────────────────────────────
    # avg chars per element, normalised to 80 chars = 1.0
    # 80 chars ≈ a meaningful sentence fragment / half a table row
    # Penalises documents where elements are mostly empty labels like "NAME: "
    if elem_count > 0:
        avg_chars = sum(len((e.get('content') or '').strip()) for e in elements) / elem_count
    else:
        avg_chars = 0.0
    content_density = min(avg_chars / 80.0, 1.0)

    # ── 2. table_cell_fill ─────────────────────────────────────────────────
    # For each table element: count cells in its HTML content.
    # Filled = cell with >1 char (excludes empty cells and lone dashes "-")
    # This is the sharpest discriminator for form extraction:
    #   Vehicle A columns in AccidentStatement return tables with all value-cells empty.
    #   Invoice and Dashboard tables are fully populated.
    # Falls back to text_fill_rate if no tables present.
    table_elems = [e for e in elements if e.get('type') == 'table']
    if table_elems:
        total_cells  = 0
        filled_cells = 0
        for t in table_elems:
            content = (t.get('content') or '')
            if '<table' in content:
                tc, fc = count_table_cells(content)
            else:
                # Non-HTML table content (rare): count non-empty tokens as proxy
                tokens = [t.strip() for t in re.split(r'[\t|]', content) if t.strip()]
                tc = len(tokens); fc = sum(1 for t in tokens if len(t) > 1)
            total_cells  += tc
            filled_cells += fc
        table_cell_fill = (filled_cells / total_cells) if total_cells > 0 else 0.0
    else:
        # No tables: use text fill rate as fallback (marks as table-free document)
        non_empty = sum(1 for e in elements if (e.get('content') or '').strip())
        table_cell_fill = (non_empty / elem_count) if elem_count > 0 else 0.0

    # ── 3. kv_extraction_rate ──────────────────────────────────────────────
    # % of text/kv-style elements that contain a label:value pair
    # where the value part has >2 non-whitespace chars.
    # This penalises "NAME: " (label captured, value lost) vs "NAME: Jane Doe" (both captured).
    # Only applied to text-type elements (not tables, figures which have their own metrics).
    text_elems  = [e for e in elements if e.get('type') in ('text', 'kv', None, 'unknown')]
    kv_elems    = [e for e in text_elems if ':' in (e.get('content') or '')]
    if kv_elems:
        kv_with_value = sum(
            1 for e in kv_elems
            if len((e.get('content') or '').split(':', 1)[-1].strip()) > 2
        )
        kv_extraction_rate = kv_with_value / len(kv_elems)
    elif elem_count > 0:
        # No KV-style elements at all: fall back to content_density proxy
        kv_extraction_rate = content_density
    else:
        kv_extraction_rate = 0.0

    # ── 4. type_diversity ──────────────────────────────────────────────────
    # Unique element types detected / 4 (capped at 1.0)
    # 4 types = full score: e.g. title + text + table + figure
    # A document parsed as all-text (missing tables/figures) scores 0.25
    unique_types  = len(set(e.get('type', 'unknown') for e in elements))
    type_diversity = min(unique_types / 4.0, 1.0)

    # ── 5. figure_description_depth ────────────────────────────────────────
    # For figure elements, avg description length / 150 (normalised to 1.0)
    # 150 chars = a substantive description vs "A diagram." (10 chars)
    # If the document contains no figures (e.g. pure text invoice): neutral 0.5
    # so it is neither rewarded nor penalised for a capability that isn't needed.
    figure_elems = [e for e in elements if e.get('type') == 'figure']
    if figure_elems:
        avg_desc_len = sum(
            len((e.get('description') or '').strip()) for e in figure_elems
        ) / len(figure_elems)
        figure_description_depth = min(avg_desc_len / 150.0, 1.0)
    else:
        figure_description_depth = 0.5  # neutral: no figures expected

    # ── Composite score ─────────────────────────────────────────────────────
    composite = (
        0.25 * content_density           +
        0.25 * table_cell_fill           +
        0.20 * kv_extraction_rate        +
        0.15 * type_diversity            +
        0.15 * figure_description_depth
    ) * 100

    # ── Element type breakdown ──────────────────────────────────────────────
    type_counts = {}
    for e in elements:
        t = e.get('type', 'unknown')
        type_counts[t] = type_counts.get(t, 0) + 1

    scorecards.append({
        'file_label'               : label,
        'filename'                 : result['filename'],
        'element_count'            : elem_count,
        'avg_chars_per_elem'       : round(avg_chars, 1),
        'content_density'          : round(content_density * 100, 1),
        'table_cell_fill'          : round(table_cell_fill * 100, 1),
        'kv_extraction_rate'       : round(kv_extraction_rate * 100, 1),
        'type_diversity'           : round(type_diversity * 100, 1),
        'figure_description_depth' : round(figure_description_depth * 100, 1),
        'composite_score'          : round(composite, 1),
        'elapsed_s'                : result['elapsed_s'],
        'element_types'            : json.dumps(type_counts),
    })

# ── Display ─────────────────────────────────────────────────────────────────
SCORE_SCHEMA = StructType([
    StructField('file_label',               StringType(),  False),
    StructField('filename',                 StringType(),  False),
    StructField('element_count',            IntegerType(), False),
    StructField('avg_chars_per_elem',       FloatType(),   False),
    StructField('content_density',          FloatType(),   False),
    StructField('table_cell_fill',          FloatType(),   False),
    StructField('kv_extraction_rate',       FloatType(),   False),
    StructField('type_diversity',           FloatType(),   False),
    StructField('figure_description_depth', FloatType(),   False),
    StructField('composite_score',          FloatType(),   False),
    StructField('elapsed_s',                FloatType(),   False),
    StructField('element_types',            StringType(),  True),
])

score_rows = [tuple(s.values()) for s in scorecards]
df_scorecard = spark.createDataFrame(score_rows, schema=SCORE_SCHEMA)
df_scorecard.createOrReplaceTempView('scorecard')

display(
    df_scorecard.select(
        'filename', 'element_count', 'avg_chars_per_elem',
        'content_density', 'table_cell_fill', 'kv_extraction_rate',
        'type_diversity', 'figure_description_depth', 'composite_score', 'elapsed_s'
    ).orderBy(F.col('composite_score').desc())
)


## 6. Format-Specific Deep Dives

Each section below:
1. Characterises what the **source document** contains
2. Computes observable **evidence** from the extracted elements
3. Prints a structured **pros / cons** analysis comparing MD output to source


### 6a. Invoice.jpg — Scanned Financial Document

**Source characteristics**
- 1-page Liberty Mutual insurance invoice (scanned JPG, ~1970s typography)
- Centred header with logo, address block, account references
- Financial table: 8 rows × 4 numeric columns (WC, GL, AUTO, TOTAL)
- DR/CR suffix notation (e.g. `2046DR`, `307CR`) — non-standard accounting markers
- No handwriting, no checkboxes, clean monospace typed font


In [0]:
# ── 6a: Invoice.jpg — Analysis ────────────────────────────────────────────
elems = parse_results['Invoice_jpg']['parsed'].get('document', {}).get('elements', [])
tables  = [e for e in elems if e.get('type') == 'table']
figures = [e for e in elems if e.get('type') == 'figure']
texts   = [e for e in elems if e.get('type') not in ('table', 'figure')]

# ── Ground-truth field list for this document ──────────────────────────────
# Defined from source image BEFORE checking extracted content.
INVOICE_FIELDS = {
    'company_name'    : 'LIBERTY MUTUAL',
    'home_office'     : 'Boston',
    'recipient_co'    : 'PHILIP MORRIS INCORPORATED',
    'recipient_name'  : 'PAUL GOLDSCHMIDT',
    'recipient_addr'  : '100 PARK AVENUE',
    'recipient_city'  : 'NEW YORK, NY  10017',
    'po_box'          : 'P. O. BOX 19038 STAMFORD, CT  06920',
    'account_number'  : '00 40 66',
    'invoice_number'  : '0125',
    'adjustment_title': 'FOURTEENTH Retrospective Premium and Dividend Adjustment',
    'effective_period': '01-01-76 TO 01-01-77',
    # Financial table rows (WC column as proxy — most distinctive values)
    'row1_wc'  : '2490252',   # Audited Premium
    'row2_wc'  : '2886168',   # Retrospective Premium
    'row3_wc'  : '2884122',   # Previously Billed Premium
    'row4_wc'  : '2046DR',    # Gross Adjustment — DR suffix
    'row5_wc'  : '432925',    # Dividend on Retro
    'row6_wc'  : '432618',    # Previously Billed Dividend
    'row7_wc'  : '307CR',     # Dividend Offset — CR suffix
    'row8_wc'  : '1739DR',    # Balance Due Company — DR suffix
    # Page metadata
    'page_ref' : 'NRD 3',
    'page_num' : '3',
}

# field checks use raw element content (HTML-escaped values would break substring search)
full_text = '\n'.join((e.get('content') or '') + ' ' + (e.get('description') or '') for e in elems)

print('=' * 65)
print(' 6a. Invoice.jpg — Extraction Analysis')
print('=' * 65)

print(f'\n  Elements total : {len(elems)}')
print(f'  Tables         : {len(tables)}')
print(f'  Figures        : {len(figures)}  ← logo not described (no AI figure element)')
print(f'  Text elements  : {len(texts)}')

# Field presence check
found = {k: v.lower() in full_text.lower() for k, v in INVOICE_FIELDS.items()}
n_found  = sum(found.values())
n_total  = len(INVOICE_FIELDS)

print(f'\n  Fields found   : {n_found} / {n_total}')
for k, v in INVOICE_FIELDS.items():
    mark = '✓' if found[k] else '✗'
    print(f'    {mark}  {k:<22} → "{v}"')

# Table cell analysis
if tables:
    for i, t in enumerate(tables):
        content = t.get('content', '')
        tc, fc = count_table_cells(content) if '<table' in content else (0, 0)
        if tc == 0:
            # Count from markdown table
            rows = [r for r in content.split('\n') if r.strip().startswith('|')]
            tc = fc = 0
            for row in rows:
                if '---' in row: continue
                cells = [c.strip() for c in row.strip().strip('|').split('|')]
                tc += len(cells); fc += sum(1 for c in cells if len(c) > 1)
        fill_pct = round(fc / tc * 100) if tc else 0
        print(f'\n  Table {i+1}: {tc} cells, {fc} filled ({fill_pct}%) — {t.get("content","")[:60]!r}...')

# DR/CR notation check
dr_cr_preserved = sum(1 for v in ['2046DR','307CR','1739DR'] if v in full_text)
print(f'\n  DR/CR markers preserved: {dr_cr_preserved}/3  ← critical accounting notation')

─────────────────────────────────────────────────────────────────
###  PROS
─────────────────────────────────────────────────────────────────
-   All header text extracted: company name, home office, mailing address
-   Recipient block complete: name, attention, street, city, state, zip
-   Account (#00 40 66) and Invoice (#0125) numbers correct
-   Financial table: all 8 rows × 4 columns present in Markdown
-   DR/CR suffix notation preserved (2046DR, 307CR, 1739DR) — non-standard markers handled correctly
-   "FOURTEENTH Retrospective Premium" title and effective period captured
-   Column headers: WC, GL, AUTO, TOTAL — correctly aligned

─────────────────────────────────────────────────────────────────
###  CONS / LIMITATIONS
─────────────────────────────────────────────────────────────────
-   Logo (Liberty Mutual torch figure) not extracted as a figure element — no AI description generated
-   No semantic structure: header, address block, table all treated as flat text elements
-   "Make check payable to Liberty Mutual and mail with a copy of this invoice to" — trailing colon lost
-   NRD 3 page code and page number "3" extracted without context (marginal notations)
-   Source is a ~1970s scan: slight blur on numbers. All values verified correct — but higher-DPI scans would further improve confidence
-   No validation of arithmetic: row 4 = row2 - row3 (2046 = 2886168 - 2884122). The model extracts values but cannot flag if they are internally inconsistent

─────────────────────────────────────────────────────────────────
### ROOT CAUSE SUMMARY
  The Invoice is the cleanest document in this set: single column,
  typed font, no handwriting, no multi-pane layout. ai_parse_document
  handles it near-perfectly. The missing logo description suggests
  that v2.0 only generates AI descriptions for explicitly figure-typed
  elements, and the logo is classified as a text/header region.

### 6b. sample_dashboard.png — KPI Dashboard Image

**Source characteristics**
- Pure PNG image — no native text layer, 100% raster
- 4 KPI headline numbers with labels and trend indicators
- 3 charts: bar (sprint velocity), pie (revenue by service), line (client satisfaction)
- 1 structured table: milestone tracker (5 rows)
- All data must be extracted from pixels via OCR + AI description


In [0]:
# ── 6b: sample_dashboard.png — Analysis ──────────────────────────────────
elems   = parse_results['dashboard_png']['parsed'].get('document', {}).get('elements', [])
figures = [e for e in elems if e.get('type') == 'figure']
tables  = [e for e in elems if e.get('type') == 'table']
texts   = [e for e in elems if e.get('type') not in ('table', 'figure')]

# ── Ground-truth KPI values (read directly from the PNG source) ────────────
# Defined from source image BEFORE checking extracted content.
DASHBOARD_FIELDS = {
    'title'               : 'TechCorp Solutions',
    'kpi_revenue'         : '38,138',
    'kpi_storypoints'     : '145 / 148',
    'kpi_client_score'    : '4.75',
    'kpi_velocity'        : '90% velocity',
    'label_revenue'       : 'Total Revenue',
    'label_storypoints'   : 'Story Points',
    'label_client'        : 'Client Score',
    'sprint5_planned'     : '31',
    'sprint5_completed'   : '36',
    'revenue_spark_dev'   : '41%',
    'revenue_ai_deploy'   : '32%',
    'revenue_dbx_setup'   : '17%',
    'satisfaction_sprint8': '4.9',
    'milestone_spark'     : 'Spark Pipelines',
    'milestone_unity'     : 'Unity Catalog',
    'milestone_status'    : 'Done',
}

# field checks use raw element content
full_text = '\n'.join((e.get('content') or '') + ' ' + (e.get('description') or '') for e in elems)

print('=' * 65)
print(' 6b. sample_dashboard.png — Extraction Analysis')
print('=' * 65)

print(f'\n  Elements total  : {len(elems)}')
print(f'  Figure elements : {len(figures)}  ← charts (bar, pie, line)')
print(f'  Table elements  : {len(tables)}   ← milestone tracker')
print(f'  Text elements   : {len(texts)}    ← KPI numbers, labels, titles')

# Field presence check
found = {k: v.lower() in full_text.lower() for k, v in DASHBOARD_FIELDS.items()}
n_found = sum(found.values())
print(f'\n  Fields found    : {n_found} / {len(DASHBOARD_FIELDS)}')
for k, v in DASHBOARD_FIELDS.items():
    mark = '✓' if found[k] else '✗'
    print(f'    {mark}  {k:<25} → "{v}"')

# Figure description depth check
print(f'\n  Figure description lengths:')
for i, fig in enumerate(figures):
    desc = (fig.get('description') or '').strip()
    has_ocr = bool((fig.get('content') or '').strip())
    print(f'    Fig {i+1}: {len(desc)} chars description | OCR data: {has_ocr}')
    print(f'           → {desc[:100]}...' if len(desc) > 100 else f'           → {desc}')




─────────────────────────────────────────────────────────────────
### PROS
─────────────────────────────────────────────────────────────────
-   All 4 KPI headline numbers extracted correctly ($38,138, 145/148, 4.75/5.0)
-   KPI labels and trend indicators present (Total Revenue, 90% velocity)
-   Bar chart: sprint velocity data extracted with planned/completed split per sprint
-   Pie chart: all 5 revenue categories with percentages (41%, 32%, 17%, 8%, 2%)
-   Line chart: all 4 sprint satisfaction scores in correct sequence
-   Milestone table: all 5 deliverables × 2 columns extracted correctly
-   AI figure descriptions provide narrative context beyond raw OCR values
-   Impressive result for a pure raster image — no native text layer to exploit

─────────────────────────────────────────────────────────────────
###  CONS / LIMITATIONS
─────────────────────────────────────────────────────────────────
-   "+12% vs 04" — should be "+12% vs Q4". The "Q" prefix was dropped (OCR ambiguity)
-   Sprint 6 Completed value is absent in the extracted data (bar may have been partially occluded)
-   "10.3 pts" extracted without context — NPS delta, not labelled in output
-   Chart data values (in code blocks) come from OCR; descriptions come from AI vision.
-      These two sources are not reconciled — a value in the code block may contradict the description
-   No confidence signal: impossible to know whether a chart value was read correctly
-      without cross-checking against the source image manually
-   The AI descriptions describe chart *shape* ("upward trend") but not all data points
-      A production pipeline would need structured JSON output, not prose descriptions

─────────────────────────────────────────────────────────────────
###  ROOT CAUSE SUMMARY
─────────────────────────────────────────────────────────────────

 The dashboard PNG is the hardest document type for any extractor
  (no text layer, complex visual encoding). The result is strong:
  headline KPIs and chart data are captured. The core limitation is
  the dual-extraction mode (OCR values + AI prose descriptions) with
  no reconciliation layer. For a production KPI ingestion pipeline,
  you would want structured JSON output from the figure description,
  not freeform prose + separate OCR code blocks.

### 6c. AccidentStatement.pdf — Multi-Column Insurance Form

**Source characteristics**
- 1-page European insurance claim form (constat amiable)
- Three-panel layout: Vehicle A (blue, left) · Circumstances (centre) · Vehicle B (yellow, right)
- Handwritten values in printed form fields (names, addresses, dates, remarks)
- 17 checkbox circumstances × 2 vehicles
- Hand-drawn accident sketch + 2 impact diagrams
- 2 handwritten signatures

> **Note**: this is the most challenging document in this benchmark. The scoring comparison  
> with LandingAI ADE is developed in detail in Part 2.


In [0]:
# ── 6c: AccidentStatement.pdf — Analysis ─────────────────────────────────
elems   = parse_results['AccidentStatement_pdf']['parsed'].get('document', {}).get('elements', [])
tables  = [e for e in elems if e.get('type') == 'table']
figures = [e for e in elems if e.get('type') == 'figure']
texts   = [e for e in elems if e.get('type') not in ('table', 'figure')]

# ── Ground-truth fields (defined from PDF source, grouped by section) ──────
# Confidence level noted: CLEAR = unambiguously legible, AMBIG = cursive uncertainty
GROUND_TRUTH = {
    # Header
    'date'              : ('09.10.2024',          'CLEAR'),
    'time'              : ('12h41',               'CLEAR'),
    'street'            : ('Limmatstrasse',        'AMBIG'),  # cursive mm/min stroke ambiguous
    'country'           : ('CH',                  'CLEAR'),
    # Checkboxes (header)
    'injury_cb_state'   : ('no [x]',                'CLEAR'),  # checkbox state captured via <input> fix
    # Witness
    'witness_name'      : ('Luke Smith',           'CLEAR'),
    'witness_email'     : ('sample@email.com',     'CLEAR'),
    # Vehicle A — handwritten (LEFT column — most likely to be dropped)
    'vA_name'           : ('Doe',                  'CLEAR'),
    'vA_firstname'      : ('Jane',                 'CLEAR'),
    'vA_address'        : ('13 Rue de la sample',  'CLEAR'),
    'vA_make'           : ('Renault 2',            'CLEAR'),
    'vA_reg'            : ('333-ABC-91',           'CLEAR'),
    'vA_driver_dob'     : ('12.01.1994',           'CLEAR'),
    'vA_driver_lic'     : ('BA12349',              'CLEAR'),
    # Vehicle A — Insurance (printed/typed — likely captured)
    'vA_insurer'        : ('CDE Assurance',        'CLEAR'),
    'vA_policy'         : ('ABC1234',              'CLEAR'),
    # Circumstances
    'circ_A4_checked'   : ('box 4 checked',        'CLEAR'),
    'circ_B5_checked'   : ('box 5 checked',        'CLEAR'),
    'circ_B9_checked'   : ('box 9 checked',        'CLEAR'),
    # Vehicle B — mostly printed/typed
    'vB_name'           : ('Smith',                'CLEAR'),
    'vB_firstname'      : ('Jade',                 'CLEAR'),
    'vB_make'           : ('Fiat 45',              'CLEAR'),
    'vB_reg'            : ('489-93-HFD',           'CLEAR'),
    'vB_insurer'        : ('ZYX Assurance',        'CLEAR'),
    'vB_policy'         : ('ZY5757',               'CLEAR'),
    # Damages & remarks
    'vA_damage'         : ('rear right bumper',    'CLEAR'),
    'vB_damage'         : ('front right bumper',   'CLEAR'),
    'vA_remarks'        : ('was on his phone',     'CLEAR'),
    'vB_remarks'        : ('turn signal',          'CLEAR'),
}

full_text = '\n'.join((e.get('content') or '') + ' ' + (e.get('description') or '') for e in elems)

print('=' * 65)
print(' 6c. AccidentStatement.pdf — Extraction Analysis')
print('=' * 65)
print(f'\n  Elements total  : {len(elems)}')
print(f'  Table elements  : {len(tables)}')
print(f'  Figure elements : {len(figures)}   ← impact diagrams + accident sketch')
print(f'  Text elements   : {len(texts)}')

# Field presence check — split by section
print('\n  Field extraction check (vs. ground truth):')
print(f'  {"Field":<25} {"Expected":<22} {"Conf":<6} {"Found?"}')
print('  ' + '─' * 60)

found_counts = {'CLEAR_found': 0, 'CLEAR_total': 0, 'AMBIG_found': 0, 'AMBIG_total': 0}
for field, (expected, conf) in GROUND_TRUTH.items():
    found = expected.lower() in full_text.lower()
    mark  = '✓' if found else '✗'
    print(f'  {mark}  {field:<25} {expected:<22} {conf:<6}')
    key = f'{conf}_total'; found_counts[key] += 1
    if found: found_counts[f'{conf}_found'] += 1

print()
ct, cf = found_counts['CLEAR_total'], found_counts['CLEAR_found']
at, af = found_counts['AMBIG_total'], found_counts['AMBIG_found']
print(f'  CLEAR fields found : {cf}/{ct} ({round(cf/ct*100)}%)')
print(f'  AMBIG fields found : {af}/{at} ({round(af/at*100) if at else 0}%)')

# Checkbox state analysis — verify [x]/[ ] now present after input-tag fix
cb_elements  = [e for e in elems if 'no' in (e.get('content') or '').lower()
                and 'yes' in (e.get('content') or '').lower()]
html_full    = doc_htmls.get('AccidentStatement_pdf', '')
ticked_boxes = html_full.count('☑')
untick_boxes = html_full.count('☐')
print(f'\n  Checkbox-like elements  : {len(cb_elements)}')
print(f'  Ticked   [x] in markdown: {ticked_boxes}  ← recovered by <input> fix')
print(f'  Unticked [ ] in markdown: {untick_boxes}')
if ticked_boxes == 0:
    print('  ⚠ Still 0 ticked boxes — run cell 7 (HTML reconstruction) before this cell')

# Two-column failure analysis
# Verify all values recovered (colspan fix + input fix)
vA_handwritten_check = ['Doe', 'Jane', '13 Rue de la sample', 'Renault 2', '333-ABC-91',
                        'BA12349', '12.01.1994']
missed = [v for v in vA_handwritten_check if v.lower() not in full_text.lower()]
print(f'\n  Vehicle A fields — post colspan-fix check: {len(vA_handwritten_check) - len(missed)}/{len(vA_handwritten_check)} recovered')
for m in missed: print(f'    ✗  "{m}"')




### AccidentStatement — Pros / Cons / Root Cause (post parser-fix analysis)

---

#### ✅ PROS

- **Header complete**: date (09.10.2024), time (12h41), country (CH), witness block
- **Vehicle A §6 insured** — Doe, Jane, 13 Rue de la sample, 4200, France — **recovered by colspan fix**
- **Vehicle A §7 vehicle** — Renault 2, 333-ABC-91, France — **recovered by colspan fix**
- **Vehicle A §8 insurance** — CDE Assurance, ABC1234, 93409, agency details — present
- **Vehicle A §9 driver** — Doe, Jane, 12.01.1994, BA12349, B, 12.01.2090 — **recovered by colspan fix**
- **Vehicle B §6 insured** — Smith, Jade, Piazza Città di Lombardia — **recovered by colspan fix**
- **Vehicle B §7 vehicle** — Fiat 45, 489-93-HFD, Italy — present
- **Vehicle B §8 insurance** — ZYX Assurance, ZY5757, 39484, piazza 230, Italy — **recovered by colspan fix**
- **Vehicle B §9 driver** — Smith Jade, 23.01.2000, 859CH, B, 12.01.2093 — present
- Both **damage descriptions**: rear right bumper / front right bumper
- Both **driver remarks**: "was on his phone!" / "Didn't have a turn signal"
- **3 figure descriptions** generated for impact diagrams and accident sketch
- Circumstances: all 17 options listed as structured text

---

#### ❌ CONS / REMAINING LIMITATIONS

- **Checkbox states captured** ✓ — `ai_parse_document` v2.0 encodes checkbox state as  
  `<input type="checkbox" checked>` / `<input type="checkbox">` inside table cell HTML.  
  The previous parser silently dropped all `<input>` tags, making this appear as a model  
  limitation. After the `<input>` fix: box 4 (Veh A ☑), boxes 5+9 (Veh B ☑) all render  
  as `[x]` / `[ ]` in the Markdown output.  
  Legally significant fields now correctly extracted.

- **Street name OCR error** — `Luminaustrasse` extracted vs `Limmatstrasse` in source.  
  Cursive handwriting, genuinely ambiguous `mm` stroke. Minor but relevant for address matching.

- **Driver name merge** in Veh B §9 — `Smith Jade` extracted as one string instead of  
  separate `NAME / First name` fields. Structural loss, not OCR error.

- **Remarks attribution** — both remarks present but not explicitly labelled by vehicle.  
  "was on his phone!" appears without a Vehicle A anchor in the output.

---

#### 🔍 ROOT CAUSE SUMMARY

The `COLUMN BLINDNESS` diagnosis from the previous version was **incorrect**.  
`ai_parse_document` v2.0 **did extract** Doe, Jane, Renault 2, BA12349 and all Vehicle B  
insurance data. They were silently discarded by the `html_table_to_md` parser.

**Root cause of data loss**: every two-column form section uses `<td colspan="2">` on the  
section title row. The old parser read that as `header_width = 1`, then `padded[:1]`  
truncated every data row to its first column (the label), dropping the value column entirely.

**Fix**: `ncols = max(len(r) for r in rows)` — column count from the widest row, not the  
first row. All 21 lost values across 5 tables are now recovered.

**Both failures were parser artefacts, not model limitations**:
- `colspan` truncation → `ncols = max(row widths)` fix
- `<input>` tag stripping → checkbox/text input rendering fix

The model extracted everything correctly. The parser lost it.


## 7. Summary — Part 1

| Document | Composite | What works well | Remaining limitation |
|---|---|---|---|
| `Invoice.jpg` | **High** | All header text, financial table (8×4), DR/CR notation | Logo not described as figure element |
| `dashboard_png` | **High** | All KPI numbers + chart data from pure raster image | OCR and AI descriptions not reconciled |
| `AccidentStatement.pdf` | **High** | All form fields + checkbox states captured after parser fixes | Street name OCR ambiguity (Lumina vs Limmat) |

> **Two parser fixes, no genuine model limitations** on AccidentStatement:  
> - **Fix 1 — `colspan`**: 21 values recovered (Veh A §6, §7, §9 + Veh B §6, §8)  
> - **Fix 2 — `<input>` tags**: checkbox states recovered (`[x]`/`[ ]`) — the model encodes  
>   ticked/unticked boxes as `<input type="checkbox" checked>` HTML inside table cells.  
>   Both were silently dropped by the parser. The model extracted everything.

> **New scoring metrics** surface real quality differences:  
> `table_cell_fill` is now accurate because `count_table_cells()` uses the colspan-aware  
> `_TableParser`. The old `_TableCellCounter` had the same truncation bug.


## 8. Save Scorecards to Delta Table

In [0]:
TABLE_NAME = f"{catalog}.{schema}.ai_parse_accuracy_part1"
run_ts     = datetime.now().isoformat()

df_to_save = df_scorecard.withColumn('run_timestamp', F.lit(run_ts))

df_to_save.write.format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable(TABLE_NAME)

print(f'Saved {df_to_save.count()} rows to {TABLE_NAME}')
display(spark.table(TABLE_NAME).orderBy('run_timestamp', 'composite_score', ascending=[False, False]))
